# Unified ML Optimizations - Analysis

This notebook visualizes:
1. Learning rate schedule (warmup + cosine decay)
2. Training loss and validation accuracy curves
3. FLOPs comparison: full-rank vs low-rank attention
4. Parameter compression ratio
5. Dynamic pruning sparsity ramp

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import math
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## 1. Learning Rate Schedule

Visualizes the cosine schedule with linear warmup used during training.

In [ ]:
from src.utils.config import ModelConfig, TrainingConfig, Config
from src.models.base_model import OptimizedTransformer
from src.training.optimizer import build_adamw_optimizer
from src.training.scheduler import get_cosine_schedule_with_warmup

warmup_steps = 100
max_steps = 1000
base_lr = 3e-4

# Dummy model for optimizer
dummy_cfg = ModelConfig(d_model=64, n_heads=4, n_layers=1, vocab_size=256, seq_len=16, rank=8)
dummy_model = OptimizedTransformer(dummy_cfg)
opt = build_adamw_optimizer(dummy_model, lr=base_lr, weight_decay=0.1)
sched = get_cosine_schedule_with_warmup(opt, warmup_steps, max_steps)

lrs = []
for _ in range(max_steps):
    lrs.append(opt.param_groups[0]['lr'])
    opt.step()
    sched.step()

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(lrs, color='steelblue', lw=1.5)
ax.axvline(warmup_steps, color='orange', ls='--', lw=1, label=f'Warmup end (step {warmup_steps})')
ax.set_xlabel('Training Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine LR Schedule with Linear Warmup')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Peak LR: {max(lrs):.2e}  |  Final LR: {lrs[-1]:.2e}')

## 2. Parameter Count: Full-Rank vs Low-Rank Attention

Compares parameter counts for full-rank vs low-rank Q/K/V projections.

In [ ]:
d_model = 512
ranks = [8, 16, 32, 64, 128, 256, 512]

full_rank_params = d_model * d_model  # per projection
low_rank_params = [2 * d_model * r for r in ranks]
compression_ratios = [full_rank_params / p for p in low_rank_params]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(['Full Rank'] + [f'r={r}' for r in ranks],
        [full_rank_params] + low_rank_params,
        color=['#e74c3c'] + ['#3498db'] * len(ranks))
ax1.set_ylabel('Parameters per Q/K/V projection')
ax1.set_title('Parameter Count vs Rank')
ax1.grid(True, alpha=0.3, axis='y')

ax2.plot(ranks, compression_ratios, 'o-', color='#2ecc71', lw=2, markersize=7)
ax2.axhline(1.0, color='gray', ls='--', lw=1)
ax2.set_xlabel('Rank r')
ax2.set_ylabel('Compression Ratio (full / low-rank)')
ax2.set_title('Compression Ratio vs Rank')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f'Full rank: {full_rank_params:,} params/projection')
print(f'r=32:      {2*d_model*32:,} params/projection  ({full_rank_params/(2*d_model*32):.1f}x compression)')

## 3. Dynamic Pruning Sparsity Ramp

In [ ]:
from src.training.optimizer import DynamicPruner

# Use a dummy nn.Linear to get the sparsity schedule
import torch.nn as nn
dummy = nn.Linear(32, 32)
pruner = DynamicPruner(dummy, target_sparsity=0.3, start_step=100, end_step=800)

steps = list(range(1000))
sparsities = [pruner._current_sparsity(s) for s in steps]

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(steps, sparsities, color='#9b59b6', lw=1.5)
ax.fill_between(steps, sparsities, alpha=0.15, color='#9b59b6')
ax.axhline(0.3, color='gray', ls='--', lw=1, label='Target sparsity 30%')
ax.set_xlabel('Training Step')
ax.set_ylabel('Sparsity (fraction zeroed)')
ax.set_title('Dynamic Magnitude Pruning Schedule')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. FLOPs Comparison: Full-Rank vs Low-Rank Model

In [ ]:
from src.utils.metrics import estimate_flops
from src.models.base_model import OptimizedTransformer
from src.utils.config import ModelConfig
import torch

device = 'cpu'

ranks_to_test = [8, 16, 32, 64, 128]
flops_list = []

for r in ranks_to_test:
    cfg = ModelConfig(d_model=256, n_heads=4, n_layers=2, vocab_size=512, seq_len=32, rank=r)
    m = OptimizedTransformer(cfg).eval().to(device)
    sample = torch.randint(0, 512, (1, 32)).to(device)
    f = estimate_flops(m, sample)
    flops_list.append(f)
    print(f'rank={r:4d}  FLOPs={f:>12,}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ranks_to_test, [f/1e6 for f in flops_list], 'o-', color='#e67e22', lw=2, markersize=8)
ax.set_xlabel('Attention Rank')
ax.set_ylabel('Forward FLOPs (millions)')
ax.set_title('FLOPs vs Attention Rank (d_model=256, 2 layers)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Quick Training Run & Loss Curves

Runs a fast training loop and plots loss + val accuracy over epochs.

In [ ]:
from src.utils.config import Config, ModelConfig, TrainingConfig
from src.data.dataset import get_dataloaders
from src.models.base_model import OptimizedTransformer
from src.training.train import train
import torch

model_cfg = ModelConfig(vocab_size=512, d_model=64, n_heads=4, n_layers=2,
                        seq_len=16, rank=8, num_classes=10)
train_cfg = TrainingConfig(batch_size=32, lr=3e-4, max_steps=200, warmup_steps=20,
                           log_interval=50,
                           device='cpu', use_amp=False)
cfg = Config(model=model_cfg, training=train_cfg)

train_loader, val_loader = get_dataloaders(32, 16, 512, 10, num_train=512, num_val=128, num_workers=0)
model = OptimizedTransformer(model_cfg)
history = train(model, train_loader, val_loader, cfg)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_losses'], 'o-', label='Train loss', color='steelblue')
ax1.plot(history['val_losses'],   's-', label='Val loss',   color='tomato')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-entropy loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['val_accs'], 'D-', color='#2ecc71')
ax2.axhline(0.1, color='gray', ls='--', lw=1, label='Random baseline (10 classes)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Accuracy')
ax2.set_title('Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

prof = history.get('profiler', {})
if prof:
    print(f'Throughput: {prof["throughput_samples_s"]:.1f} samples/s over {prof["steps"]} steps')